# M04. 컬럼 정리 (rename, drop, str.strip)

> 📌 **언제 필요한가**  
> 컬럼명에 단위/공백이 박혀 있거나, 필요 없는 컬럼이 많거나, 컬럼명을 깔끔하게 정리하고 싶을 때.

## 이 모듈에서 배울 것

- 필요한 컬럼만 추출 (`df[['col1', 'col2']]`)
- 컬럼명 변경 (`rename`, `df.columns = `)
- 컬럼 제거 (`drop(columns=)`)
- 공백/단위 처리 (`str.strip`, `str.replace`)
- 컬럼명 공백 함정 — merge가 안 될 때 #1 원인

---


## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.
> 받는 곳 링크를 누르면 바로 받으러 갈 수 있어요.

- `교육부_시도별 진로전담교사 배치현황_20241231.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15097012/fileData.do)
- `교육부_지방교육재정정보_순계_2024.csv` — 인코딩 `cp949` — [공공데이터포털](https://www.data.go.kr/data/15052877/fileData.do) → [지방교육재정알리미](https://www.eduinfo.go.kr/portal/open/openData/dataSetPage.do) (예산통합공시 > 재정규모 > 통합재정 > 순계)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. 필요한 컬럼만 추출하기

받은 데이터에 컬럼이 너무 많을 때, 분석에 필요한 것만 골라서 작은 DataFrame 만들기.


In [ ]:
import pandas as pd

counselor = pd.read_csv('data/교육부_시도별 진로전담교사 배치현황_20241231.csv', encoding='cp949')
print(f"원본 컬럼: {list(counselor.columns)}")
print(f"shape: {counselor.shape}")
counselor.head()


In [ ]:
# 시도, 진로전담교사수, 배치율만 추출
counselor_small = counselor[['시도', '진로전담교사수', '전체학교대비_배치율']]
print(f"추출 후 shape: {counselor_small.shape}")
counselor_small.head()


> 💡 **대괄호 두 개** `df[[...]]`: 컬럼 이름 리스트로 추출 → DataFrame 반환  
> **대괄호 하나** `df['시도']`: 컬럼 하나 → Series 반환


## 2. 컬럼명 변경 — `rename`


In [ ]:
# 방법 1: rename으로 일부만 변경
counselor_renamed = counselor_small.rename(columns={
    '전체학교대비_배치율': '배치율'
})
counselor_renamed.head()


In [ ]:
# 방법 2: df.columns로 전부 한꺼번에 변경
counselor_renamed2 = counselor_small.copy()
counselor_renamed2.columns = ['region', 'teachers', 'ratio']
counselor_renamed2.head()


## 3. 컬럼 제거 — `drop`


In [ ]:
# 받은 그대로 읽기 — 맨 끝에 빈 'Unnamed: 4' 컬럼이 딸려옴 (줄 끝 쉼표 때문)
budget = pd.read_csv('data/교육부_지방교육재정정보_순계_2024.csv', encoding='cp949')
print("원본 컬럼:", list(budget.columns))
budget.head(3)


In [ ]:
# 분석에 필요 없는 빈 'Unnamed: 4' 컬럼 제거
budget_clean = budget.drop(columns=['Unnamed: 4'])
print("제거 후 컬럼:", list(budget_clean.columns))


## 4. 공백 처리 — `str.strip()` ⭐

**한국 공공데이터의 흔한 함정**: 컬럼명이나 값에 보이지 않는 공백이 끼어 있음.


In [ ]:
# 진짜 어이없는 함정 사례
budget = pd.read_csv('data/교육부_지방교육재정정보_순계_2024.csv', encoding='cp949')

# 함정 1: 컬럼명에 앞 공백이 박혀 있음
print("컬럼명 repr:", [repr(c) for c in budget.columns])
print()
print("budget['지역구분'] 으로 접근되나? →", '지역구분' in budget.columns)   # False!
print("실제 이름은 ' 지역구분'(앞 공백) →", ' 지역구분' in budget.columns)    # True


**보이세요?** `' 지역구분'`, `' 항목구분'`, `' 금액(원)'` — 컬럼명 **앞**에 공백이 한 칸씩 붙어 있어요.

그래서 `budget['지역구분']`은 `KeyError`가 납니다. 눈에 안 보이니 더 골치 아픈 함정이죠.

**해결**: 컬럼명 전체에 `str.strip()`을 한 번 걸어줍니다.

In [ ]:
# 컬럼명 공백부터 제거
budget.columns = budget.columns.str.strip()
print("정리 후 컬럼:", list(budget.columns))
print()

# 함정 2: 값에도 앞 공백이 있음
print("지역구분 첫 값들 (공백 보이게 repr):")
for val in budget['지역구분'].head():
    print(f"  {repr(val)}")   # ' 서울', ' 부산' ...


In [ ]:
# 값 공백 때문에 비교가 안 됨
print(budget['지역구분'][0] == '서울')   # False! (' 서울' 이라서)
print(repr(budget['지역구분'][0]))       # ' 서울' ← 앞 공백

# str.strip()으로 값 공백 제거
budget['지역구분'] = budget['지역구분'].str.strip()
print(budget['지역구분'][0] == '서울')   # True!


> 💡 **`str.strip()` 룰**: 받은 데이터는 **컬럼명**과 **문자열 값** 둘 다에 strip을 걸어두는 게 안전해요. 손해 볼 일 없음.
> ```python
> df.columns = df.columns.str.strip()                             # 컬럼명
> for c in df.select_dtypes('object'): df[c] = df[c].str.strip()  # 값
> ```

## 5. 단위 제거 — `str.replace`

컬럼명이나 값에 단위가 박혀 있을 때.

```
"분석대상자수 (명)" → "분석대상자수"
"인지율 (%)" → "인지율"
```


In [ ]:
import pandas as pd

# 시리즈에 단위가 박힌 경우
example = pd.Series(['분석대상자수 (명)', '인지율 (%)', '표준오차'])
print("원본:", list(example))

cleaned = example.str.replace(r' \(.*\)', '', regex=True)
print("정리:", list(cleaned))


## 6. 본인 데이터에 적용해보기 ✏️


In [ ]:
# 본인 데이터로
# my_df = pd.read_csv('파일.csv', encoding='cp949')
# 
# # Step 1: 컬럼 추출
# my_df = my_df[['필요한_컬럼1', '필요한_컬럼2']]
# 
# # Step 2: 컬럼명 정리
# my_df = my_df.rename(columns={'긴이름': '짧은이름'})
# 
# # Step 3: 모든 문자열 컬럼에 strip 적용 (안전 룰)
# for col in my_df.select_dtypes(include='object').columns:
#     my_df[col] = my_df[col].str.strip()


## 7. ⚠️ 함정 / 주의사항

### 7.1 컬럼명에 공백/줄바꿈
받은 데이터의 컬럼명에 `'시도 '`처럼 공백이 있으면 평범하게 접근이 안 됨.
```python
df.columns = df.columns.str.strip()  # 컬럼명도 strip
```

### 7.2 같은 의미 다른 표기
`'서울'` vs `'서울특별시'` vs `'서울 특별시'` ← merge 안 됨.  
표준 명칭 매핑 사전 만들기:
```python
mapping = {'서울': '서울특별시', '서울 특별시': '서울특별시'}
df['시도'] = df['시도'].replace(mapping)
```

### 7.3 NaN에 str 메서드
`Series.str.strip()`은 NaN을 그대로 NaN으로 둠 (에러 안 남). OK.


## 8. 📚 더 알아보기

- `df.columns.str.lower()` — 컬럼명 소문자로
- `df.columns.str.replace(' ', '_')` — 공백을 언더스코어로
- `df.add_prefix('teacher_')` — 모든 컬럼명에 접두사
- `df.filter(like='_rate')` — 이름에 특정 문자열 포함된 컬럼만
